# 13 · The Phase 2 student run, end to end (GPU T4 x2)

One committed session does every GPU step of the student lane, through the pipeline's single
entry point (`python -m src.pipeline.run`, work plan v4 §5.1), for every arm:

1. **Evaluation tables** from the two competitions, by the id lists in git (`src/student/eval_sets.py`):
   gate 63,978 · H2 10,000 · bias 21,577 · obfuscation 3,000 (the data pipeline's own script).
2. **Eight arms**, two at a time (one per T4), each in its own process with the tuned values
   (trial 26, `docs/tuning.md`): train, the §2.5 package (ONNX), prediction files for val, gate,
   H2, bias and obfuscation, gold files, every metric, the gate and bias report, per-run plots.
3. **CPU latency** of every package (batch 1, 2 threads), measured with nothing else running.
4. **Selection** (`select_best.py`): bias check and latency budget, then the best validation AUC.
5. **Truncation sweep** of the selected model on the gate set (64, 128, 256 tokens).
6. **Results** (`src/eval/results.py`): H1 / H2 tables with 1,000-resample bootstrap intervals,
   paired comparisons (mixed vs gold, stack vs TF-IDF, ...), the leak check, obfuscation recall,
   and the results plots.
7. **`phase2_results.zip`**: everything small (ids, scores, labels, metrics, plots; no comment
   text, no weights). The weights and packages stay in this version's output.

| GPU 0 | GPU 1 |
|---|---|
| distilbert-mixed (63 min) | distilbert-gold (63) |
| minilm-mixed (52) | distilbert-stack (63) |
| modernbert-mixed (210) | distilbert-tfidf (63) |
| | distilbert-calqwen (63) |
| | distilbert-mixed-5ep (78) |

About 6 to 7 hours in all. An arm starts only if it fits `BUDGET_H` (with time kept for the steps
after training), so the session ends cleanly. With one GPU the arms run in one queue in the
order of need and the last ones may be skipped.

**Before running:**
1. Accept the rules of both competitions (their **Data** tab, "I Understand and Accept"):
   `jigsaw-toxic-comment-classification-challenge`, `jigsaw-unintended-bias-in-toxicity-classification`.
2. **Add Input → Competitions**: add both.
3. **The soft labels**, one of: **Add Input → Your Datasets →** the private dataset with the
   bucket's `soft_labels/` and `tfidf/` folders (found automatically, no secret needed); or
   **Add-ons → Secrets**: `GCS_TEACHER_KEY` (the service-account JSON); or `HF_TOKEN` with
   `EXTRA = ["storage.backend=hf"]`. Optional: `MLFLOW_TRACKING_URI`,
   `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD` (the team's MLflow server).
4. **Settings**: Accelerator **GPU T4 x2**, Internet **On**, Persistence: none needed.
5. **Save Version → Save & Run All (Commit)**. Not the interactive session: its outputs are not kept.

In [ ]:
# Code version: the commit with src/pipeline/run.py, select_best.py and src/eval/results.py.
REPO_URL = "https://github.com/Nishaant-Soni/Distillate.git"
REPO_REF = "80a4502d97d882ddf7de90c87cd3db743bcedb36"

BUDGET_H = 11          # wall-clock hours this session may use (Kaggle stops at 12)
POST_MIN = 50          # minutes kept for latency, selection, truncation, results and the zip

# Trial 26 of the merged study (28 trials): val mean AUC 0.99364 (docs/tuning.md).
TUNED = ["loss.alpha=0.3432987447660936", "loss.temperature=3.71295432376268",
         "train.lr=9.22542696771891e-05", "train.epochs=4", "model.max_length=256"]
GRID = {
    "distilbert-mixed":     [],
    "distilbert-gold":      ["loss.alpha=1.0"],
    "distilbert-stack":     ["loss.alpha=0.0"],
    "distilbert-tfidf":     ["soft_labels.arm=tfidf"],
    "distilbert-calqwen":   ["soft_labels.arm=calqwen"],
    "distilbert-mixed-5ep": ["train.epochs=5"],          # the check outside the study
    "minilm-mixed":         ["model.name=minilm"],
    "modernbert-mixed":     ["model.name=modernbert"],
}
# Minutes per arm (the first ablation run: training, plus scoring and the package).
EST_MIN = {"distilbert-mixed": 63, "distilbert-gold": 63, "distilbert-stack": 63,
           "distilbert-tfidf": 63, "distilbert-calqwen": 63, "distilbert-mixed-5ep": 78,
           "minilm-mixed": 52, "modernbert-mixed": 210}
QUEUES = {0: ["distilbert-mixed", "minilm-mixed", "modernbert-mixed"],
          1: ["distilbert-gold", "distilbert-stack", "distilbert-tfidf", "distilbert-calqwen",
              "distilbert-mixed-5ep"]}
ONE_GPU_ORDER = ["distilbert-mixed", "distilbert-gold", "distilbert-stack", "distilbert-tfidf",
                 "distilbert-calqwen", "distilbert-mixed-5ep", "minilm-mixed", "modernbert-mixed"]
SLOWDOWN = 1.15        # two trainings share 4 CPUs

LATENCY_THREADS = 2    # the serving target: batch 1 on 2 vCPUs
LATENCY_REQUESTS = 300
TRUNCATION = [64, 128, 256]
BOOTSTRAP = 1000
EXTRA = []             # e.g. ["storage.backend=hf"] with an HF_TOKEN secret instead of GCS
STORAGE = None         # (cell 4 reads OPTUNA_STORAGE into it; unused here)

import time
SESSION_START = time.perf_counter()

In [ ]:
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "iterative-stratification",
                "mlflow", "optuna==5.0.0", "google-cloud-storage", "onnx", "onnxscript",
                "onnxruntime"], check=True)

REPO_DIR = Path("/tmp/Distillate")   # outside /kaggle/working, so the repo is not saved as output
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", REPO_REF], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
MP = REPO_DIR / "Model-Pipeline"
sys.path.insert(0, str(MP))
print("code_commit:", CODE_COMMIT)
assert (MP / "src" / "pipeline" / "run.py").exists(), "REPO_REF has no src/pipeline/run.py"

In [ ]:
import torch
import transformers

N_GPU = torch.cuda.device_count()
print("torch", torch.__version__, "| transformers", transformers.__version__, "| python",
      sys.version.split()[0])
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
assert N_GPU >= 1, "no GPU: set the accelerator to GPU T4 x2"
if N_GPU == 1:
    QUEUES = {0: ONE_GPU_ORDER}
    print("one GPU: all arms in one queue; the last ones may not fit BUDGET_H")

In [ ]:
# Secrets, into the environment (the arm processes inherit it). Never printed.
import glob
import os

from kaggle_secrets import UserSecretsClient

for name in ("GCS_TEACHER_KEY", "HF_TOKEN", "MLFLOW_TRACKING_URI",
             "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    try:
        os.environ[name] = UserSecretsClient().get_secret(name)
    except Exception:   # secret not attached
        pass
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# No secret needed if the soft labels are attached as a Kaggle dataset: a copy of the bucket's
# soft_labels/ and tfidf/ folders (ids and scores only). The same checksums are checked.
LOCAL_STORE = None
found = sorted(glob.glob("/kaggle/input/**/soft_labels/stack-v1", recursive=True))
if found:
    LOCAL_STORE = Path(found[0]).parent.parent
    assert (LOCAL_STORE / "tfidf" / "tfidf-lr-v1-oof" / "tfidf_oof.parquet").exists(), \
        f"{LOCAL_STORE}: the dataset has soft_labels/ but not tfidf/tfidf-lr-v1-oof/"
    EXTRA = [*EXTRA, f"soft_labels.local_root={LOCAL_STORE}"]
print("store key:", "GCS_TEACHER_KEY" in os.environ, "| HF token:", "HF_TOKEN" in os.environ,
      "| soft-label dataset:", LOCAL_STORE, "| MLflow server:", "MLFLOW_TRACKING_URI" in os.environ)
assert LOCAL_STORE or "GCS_TEACHER_KEY" in os.environ or "HF_TOKEN" in os.environ, \
    "attach the soft-label dataset (Add Input) or the GCS_TEACHER_KEY secret (Add-ons -> Secrets)"

## 1. Training data and the evaluation tables (text in /tmp only)

In [ ]:
import glob
import json

import pandas as pd

from src.student.eval_sets import build_eval_tables
from src.teacher.config import load_config as load_teacher_config
from src.teacher.data import check_frame, find_competition_dir, read_competition_split, to_raw_frame

OUT = Path("/kaggle/working")
DATA = Path("/tmp/wiki")              # text stays in /tmp: never in the output
PROCESSED = Path("/tmp/processed")
DATA.mkdir(parents=True, exist_ok=True)
RAW_TRAIN = DATA / "wiki_train_raw.parquet"

comp_dir = find_competition_dir("/kaggle/input")   # the Toxic Comment competition
raw = to_raw_frame(read_competition_split(comp_dir, "train"))
print("raw train:", check_frame(raw, load_teacher_config()["data"]["expected_rows"]["train"]))
raw.to_parquet(RAW_TRAIN, index=False)
del raw

civil = sorted(glob.glob("/kaggle/input/**/test_private_expanded.csv*", recursive=True))
assert len(civil) == 1, f"attach jigsaw-unintended-bias-in-toxicity-classification, found {civil}"
EVAL_INFO = build_eval_tables(comp_dir, civil[0], PROCESSED)
print(json.dumps(EVAL_INFO["rows"], indent=1))
expected = {"wikipedia/gate.parquet": 63978, "civil/h2_eval.parquet": 10000,
            "civil/bias_eval.parquet": 21577}
for rel, n in expected.items():
    assert EVAL_INFO["rows"][rel] == n, (rel, EVAL_INFO["rows"][rel], n)
(OUT / "eval_tables.json").write_text(json.dumps(EVAL_INFO, indent=2) + "\n")   # counts only

In [ ]:
# Settings every arm shares. Latency is measured after training, alone on the CPU.
def base(gpu):
    return [
        f"paths.raw_train={RAW_TRAIN}",
        f"paths.split_ids_dir={REPO_DIR / 'Data-Pipeline' / 'data' / 'splits'}",
        f"paths.out_dir={OUT / 'student'}",
        f"paths.mlflow_db={OUT / 'mlruns' / f'gpu{gpu}.db'}",   # one SQLite file per GPU
        f"paths.mlflow_artifacts={OUT / 'mlruns' / 'artifacts'}",
        f"paths.reports_dir={OUT / 'reports' / f'gpu{gpu}'}",
        f"paths.cache_dir=/tmp/cache/gpu{gpu}",
        "data.source=ids",
        f"pipeline.data_dir={PROCESSED}",
        "pipeline.latency=false",
        "package.enabled=true",
        "mlflow.experiment=distillate-student-ablation",
        *EXTRA,
    ]


def arm_sets(name, gpu):
    return base(gpu) + TUNED + [f"run_name={name}", *GRID[name]]


from src.student.config import load_config

for name in GRID:                      # every arm's config loads before any GPU time is spent
    load_config(None, arm_sets(name, 0))
print("configs ok:", list(GRID))

## 2. Train, score and check every arm (two GPUs)

In [ ]:
# Two queues, one per GPU; each arm is one `python -m src.pipeline.run` process.
import threading

LOGS = OUT / "logs"
LOGS.mkdir(exist_ok=True)
status = {}


def minutes_used():
    return (time.perf_counter() - SESSION_START) / 60


def run_queue(gpu, arms):
    for name in arms:
        need = EST_MIN[name] * (SLOWDOWN if len(QUEUES) > 1 else 1.0)
        if minutes_used() + need + POST_MIN > BUDGET_H * 60:
            status[name] = {"gpu": gpu, "state": "skipped",
                            "why": f"{minutes_used():.0f} + {need:.0f} + {POST_MIN} min > "
                                   f"BUDGET_H {BUDGET_H} h"}
            continue
        status[name] = {"gpu": gpu, "state": "running", "start": time.perf_counter()}
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
        with open(LOGS / f"{name}.log", "w") as log:
            proc = subprocess.run([sys.executable, "-m", "src.pipeline.run", "--set",
                                   *arm_sets(name, gpu)], cwd=MP, env=env, stdout=log,
                                  stderr=subprocess.STDOUT)
        status[name].update(state="done" if proc.returncode == 0 else f"exit {proc.returncode}",
                            minutes=round((time.perf_counter() - status[name]["start"]) / 60, 1))


def last_line(name):
    try:
        lines = (LOGS / f"{name}.log").read_text(errors="replace").strip().splitlines()
        return lines[-1][-150:] if lines else ""
    except OSError:
        return ""


threads = [threading.Thread(target=run_queue, args=(g, arms), daemon=True)
           for g, arms in QUEUES.items()]
for t in threads:
    t.start()
while any(t.is_alive() for t in threads):
    time.sleep(600)
    print(f"--- {minutes_used():.0f} min")
    for name, s in status.items():
        extra = last_line(name) if s["state"] == "running" else s.get("minutes", s.get("why", ""))
        print(f"  gpu{s['gpu']} {name:22s} {s['state']:8s} {extra}")
for t in threads:
    t.join()
print(json.dumps({k: {x: v for x, v in s.items() if x != "start"} for k, s in status.items()},
                 indent=1))
(OUT / "arm_status.json").write_text(json.dumps(
    {k: {x: v for x, v in s.items() if x != "start"} for k, s in status.items()}, indent=2))

In [ ]:
# The finished runs, and any step that failed inside one (the log says why).
RUNS = {}
for path in sorted((OUT / "student").glob("*/pipeline.json")):
    record = json.loads(path.read_text())
    arm = record["run_name"].rsplit("-", 1)[0]
    RUNS[arm] = Path(record["run_dir"])
    errors = [k for k in record["errors"] if not k.endswith("_traceback")]
    head = record.get("headline") or {}
    print(f"{arm:22s} val {record['best_val_mean_auc']:.5f} | gate "
          f"{head.get('gate', {}).get('mean_auc', float('nan')):.5f} | h2 "
          f"{head.get('h2_eval', {}).get('mean_auc', float('nan')):.5f} | bias ok "
          f"{(record.get('gate') or {}).get('bias_passed')} | errors {errors or 'none'}")
missing = [a for a in GRID if a not in RUNS]
print("missing arms:", missing or "none")
for arm in missing:
    if (LOGS / f"{arm}.log").exists():
        print(f"\n{arm} log tail:\n" + "\n".join(
            (LOGS / f"{arm}.log").read_text(errors="replace").splitlines()[-15:]))

## 3. CPU latency, alone on the machine

In [ ]:
# Batch-1 CPU latency of every package, one at a time (nothing else running).
from src.student.evaluate import measure_latency

val_ids = set((REPO_DIR / "Data-Pipeline" / "data" / "splits" / "val_ids.txt").read_text().split())
val_texts = pd.read_parquet(RAW_TRAIN, columns=["id", "text"])
val_texts = val_texts[val_texts["id"].isin(val_ids)]["text"].tolist()
for arm, run_dir in RUNS.items():
    package = run_dir / "package"
    if not (package / "model.onnx").exists():
        print(arm, "no package:", json.loads((run_dir / "summary.json").read_text())["package"])
        continue
    try:
        lat = measure_latency(package, val_texts, LATENCY_THREADS, LATENCY_REQUESTS)
        (run_dir / "latency.json").write_text(json.dumps(lat, indent=2) + "\n")
        print(f"{arm:22s} p50 {lat['p50_ms']:6.1f}  p95 {lat['p95_ms']:6.1f}  p99 "
              f"{lat['p99_ms']:6.1f} ms  ({lat['cpu']}, {lat['threads']} threads)")
    except Exception as e:
        print(arm, "latency FAILED:", type(e).__name__, e)

## 4. Selection rule

In [ ]:
from src.student import select_best

SEL_CFG = load_config(None, base(0))
selection = select_best.run(list(RUNS.values()), OUT / "selection", SEL_CFG, log_mlflow=True)
print((OUT / "selection" / "selection.md").read_text())
SELECTED = (selection["selected"] or {}).get("arm")

## 5. Truncation sweep (sensitivity to input length)

In [ ]:
# Truncation sweep of the selected model (and the tuned mixed recipe) on the gate set.
from src.student.evaluate import truncation_sweep

gate_table = pd.read_parquet(PROCESSED / "wikipedia" / "gate.parquet", columns=["id", "text"])
for arm in dict.fromkeys([a for a in (SELECTED, "distilbert-mixed") if a in RUNS]):
    try:
        print(arm, truncation_sweep(RUNS[arm], gate_table, TRUNCATION))
    except Exception as e:
        print(arm, "truncation FAILED:", type(e).__name__, e)
del gate_table
torch.cuda.empty_cache()

## 6. Results across arms: H1, H2, comparisons, plots

In [ ]:
from src.eval import results as results_mod

RESULTS = results_mod.build(list(RUNS.values()), OUT / "results", BOOTSTRAP, workers=4,
                            selected=SELECTED)
print((OUT / "results" / "results.md").read_text())

## 7. The selected package and the results zip

In [ ]:
# The selected student's §2.5 package, checked, under one obvious name.
import shutil

from src.student.package import verify_package

if SELECTED:
    pkg = RUNS[SELECTED] / "package"
    problems = verify_package(pkg)
    print("package check:", problems or "ok")
    shutil.copytree(pkg, OUT / "selected_package", dirs_exist_ok=True)
    meta = json.loads((pkg / "meta.json").read_text())
    print({k: meta.get(k) for k in ("model_version", "model_id", "max_length", "alpha",
                                    "temperature", "teacher_version", "code_commit")})

In [ ]:
# One zip of everything small: ids, scores, labels, metrics, reports, plots, logs.
# No comment text (the tables with text are in /tmp), no weights, no ONNX files.
import zipfile

SMALL = {"summary.json", "metrics.json", "pipeline.json", "scoring.json", "latency.json",
         "meta.json", "thresholds.json", "thresholds_report.json", "confusion_matrices.json"}
with zipfile.ZipFile(OUT / "phase2_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for arm, run_dir in RUNS.items():
        for p in sorted(run_dir.rglob("*")):
            rel = p.relative_to(run_dir)
            if not p.is_file() or rel.parts[0] in ("model", "package", "splits", "gold") and \
                    p.name not in ("meta.json", "thresholds.json", "thresholds_report.json"):
                continue
            if p.suffix in (".onnx", ".safetensors", ".bin", ".pt"):
                continue
            if p.name in SMALL or rel.parts[0] in ("predictions", "eval", "plots"):
                z.write(p, f"runs/{arm}/{rel}")
    first = next(iter(RUNS.values()), None)
    if first is not None:                                 # the gold files are the same for all
        for p in sorted((first / "gold").glob("*.parquet")):
            z.write(p, f"gold/{p.name}")
    for folder in ("selection", "results", "logs"):
        for p in sorted((OUT / folder).rglob("*")):
            if p.is_file():
                z.write(p, f"{folder}/{p.relative_to(OUT / folder)}")
    for name in ("eval_tables.json", "arm_status.json"):
        if (OUT / name).exists():
            z.write(OUT / name, name)
    z.writestr("run_info.json", json.dumps({
        "code_commit": CODE_COMMIT, "tuned": TUNED, "selected": SELECTED,
        "hours": round(minutes_used() / 60, 2), "gpus": N_GPU}, indent=2))
size = (OUT / "phase2_results.zip").stat().st_size / 1e6
print(f"phase2_results.zip: {size:.1f} MB, {len(zipfile.ZipFile(OUT / 'phase2_results.zip').namelist())} files")
print(f"session: {minutes_used() / 60:.2f} h")

In [ ]:
total = 0
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        total += p.stat().st_size
print(f"/kaggle/working: {total / 1e9:.2f} GB")
for p in sorted(OUT.iterdir()):
    print(" ", p.name)